# 🛍️ Perilaku Belanja Online Orang Indonesia di E-Commerce (Tokopedia)
### Produk Terlaris & Pengaruh Promo

**Sebuah analisis data eksploratif (EDA) untuk menjawab 6 pertanyaan bisnis**

| | |
|---|---|
| **Disusun oleh** | Imanuel R. — *ganti dengan nama & link LinkedIn kamu* |
| **Tools** | Python · pandas · matplotlib · seaborn · SciPy · pytrends |
| **Lingkungan** | Google Colab |
| **Sumber data** | *Tokopedia Product and Review Dataset* (Kaggle, 2025) + Google Trends Indonesia |
| **Status** | Tahap 1 — Persiapan ✅ |

## 📌 Latar Belakang

E-commerce sudah jadi bagian sehari-hari orang Indonesia. Saat Harbolnas atau tanggal kembar
(11.11, 12.12) tiba, timeline kita penuh banner diskon. Tapi pertanyaannya:

> **Apakah diskon besar benar-benar bikin produk lebih laku? Kategori apa yang paling
> "cinta promo", dan kategori apa yang pembelinya cuek saja?**

Proyek ini mencoba menjawab itu dengan data produk Tokopedia yang tersedia publik, lalu
melengkapinya dengan data minat pencarian Google Trends untuk melihat pola musiman.

Hasilnya ditujukan untuk **penjual online dan tim marketing** yang ingin tahu:
di mana uang promo paling efektif dibelanjakan, dan di mana justru sia-sia.

## ❓ Enam Pertanyaan Bisnis

1. **Kategori produk apa yang paling banyak terjual?**
2. **Di rentang harga berapa produk paling laku?**
3. **Seberapa besar pengaruh diskon terhadap penjualan?** Apakah diskon 10% berbeda efeknya dengan 50%?
4. **Seberapa besar pengaruh rating terhadap penjualan?**
5. **Kategori mana yang paling sensitif terhadap promo**, dan mana yang tidak terlalu peduli promo?
6. **Bagaimana tren minat masyarakat saat tanggal kembar (11.11, 12.12), Harbolnas, dan promo besar lainnya?**

## 🗂️ Daftar Isi

- **Tahap 1 — Persiapan**
  - 1.1 Install library
  - 1.2 Import library & pengaturan tampilan
  - 1.3 Memilih dataset
  - 1.4 Cara mengunggah dataset ke Colab
  - 1.5 Membaca data pertama kali
- **Tahap 2 — Memahami & Membersihkan Data**
- **Tahap 3 — Analisis per Pertanyaan** (Q1–Q6)
- **Tahap 4 — Kesimpulan & Insight**
- **Tahap 5 — Output Portofolio**

## ⚠️ Catatan Penting Sebelum Mulai (Mohon Dibaca)

Tiga hal ini berlaku untuk **seluruh** analisis di notebook ini:

1. **Kolom "jumlah terjual" adalah PROXY, bukan angka penjualan resmi.**
   Tokopedia menampilkan jumlah terjual dalam bentuk kasar ("1rb+ terjual"), dan angka itu
   hanya terlihat di halaman produk. Artinya ini **perkiraan minat beli**, bukan laporan
   penjualan dari perusahaan.

2. **Semua temuan bersifat KORELASI, bukan sebab-akibat.**
   Kalimat "produk berdiskon lebih laku" **tidak** berarti "diskon menyebabkan produk laku".
   Bisa jadi sebaliknya: produk yang memang sudah laku sengaja didiskon oleh penjualnya.

3. **Tidak ada scraping ke Tokopedia.**
   Sesuai Ketentuan Layanan mereka, proyek ini hanya memakai dataset publik yang sudah
   tersedia (Kaggle), dan Google Trends untuk data tren. Tidak ada satu pun request
   yang dikirim ke halaman produk Tokopedia oleh notebook ini.

---
# 🧰 Tahap 1 — Persiapan

**Tujuan tahap ini:** menyiapkan "dapur" sebelum memasak.

Yang akan kita lakukan:
1. Install library tambahan yang belum ada di Colab.
2. Import semua library dan mengatur gaya grafik supaya rapi.
3. Memilih dataset dan memahaminya sekilas.
4. Mengunggah dataset ke Colab.

Di akhir tahap ini kita sudah punya tabel data yang siap dibersihkan di Tahap 2.

---

### 1.1 Install Library

Colab sudah punya `pandas`, `numpy`, `matplotlib`, `seaborn`, dan `scipy`.
Kita hanya perlu menambah dua library:

| Library | Kegunaan |
|---|---|
| `pytrends` | Mengambil data **Google Trends** (dipakai di Pertanyaan 6) |
| `kagglehub` | Mengunduh dataset Kaggle langsung dari Colab (opsional, alternatif upload manual) |

> Catatan: `pytrends` adalah library tidak resmi. Google kadang memblokir permintaannya
> (error `429 Too Many Requests`). Kalau itu terjadi, notebook ini menyediakan jalur
> cadangan berupa file CSV hasil ekspor manual dari situs Google Trends. Kita bahas nanti
> di Pertanyaan 6.

In [1]:
import sys

# Cek apakah notebook ini dijalankan di Google Colab
DI_COLAB = "google.colab" in sys.modules

if DI_COLAB:
    # Tanda "!" berarti menjalankan perintah terminal, bukan Python.
    # "-q" = quiet, supaya output instalasi tidak terlalu panjang.
    !pip install -q pytrends kagglehub
    print("✅ Library tambahan selesai di-install.")
else:
    print("ℹ️ Bukan Colab. Kalau ada library yang belum ada, jalankan:")
    print("   pip install -r requirements.txt")

ℹ️ Bukan Colab. Kalau ada library yang belum ada, jalankan:
   pip install -r requirements.txt


### 1.2 Import Library & Pengaturan Tampilan

Semua library yang dipakai sepanjang notebook di-import di satu tempat supaya mudah dicek.
Di sel yang sama kita juga mengatur **tema grafik**: warna hijau Tokopedia untuk elemen
utama, oranye untuk hal yang berkaitan dengan promo.

Menyamakan gaya grafik dari awal penting untuk portofolio — recruiter menilai konsistensi
visual hampir secepat mereka membaca angkanya.

In [2]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scipy
from scipy import stats

# Sembunyikan peringatan teknis yang tidak relevan untuk pembaca
warnings.filterwarnings("ignore")

# Pengaturan tampilan tabel
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ------------------------------------------------------------------
# Palet warna & tema grafik
# ------------------------------------------------------------------
WARNA_UTAMA  = "#03AC0E"   # hijau Tokopedia -> elemen utama
WARNA_AKSEN  = "#F59E0B"   # oranye -> penekanan / promo
WARNA_NETRAL = "#4B5563"   # abu-abu -> pembanding
PALET = ["#03AC0E", "#0EA5E9", "#F59E0B", "#EF4444", "#8B5CF6", "#14B8A6", "#9CA3AF"]

sns.set_theme(style="whitegrid", palette=PALET)
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#D1D5DB",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# Folder untuk menyimpan grafik (dipakai di README GitHub nanti)
FOLDER_GAMBAR = Path("images")
FOLDER_GAMBAR.mkdir(parents=True, exist_ok=True)

print(f"pandas {pd.__version__} | numpy {np.__version__} | "
      f"matplotlib {plt.matplotlib.__version__} | seaborn {sns.__version__} | "
      f"scipy {scipy.__version__}")
print("✅ Semua library berhasil di-import.")

pandas 3.0.6 | numpy 2.5.4 | matplotlib 3.11.2 | seaborn 0.13.2 | scipy 1.18.1
✅ Semua library berhasil di-import.


Dua fungsi bantu yang akan sering dipakai:

- `ke_rupiah()` — mengubah angka `1500000` menjadi `"Rp1,5 jt"` supaya label grafik tidak sesak.
- `simpan_grafik()` — menyimpan grafik ke folder `images/` agar bisa dipakai di README.

In [3]:
def ke_rupiah(nilai, ringkas: bool = True) -> str:
    '''Format angka menjadi teks rupiah. Contoh: 1500000 -> 'Rp1,5 jt'.'''
    if pd.isna(nilai):
        return "-"
    if ringkas:
        if abs(nilai) >= 1_000_000_000:
            return f"Rp{nilai / 1_000_000_000:.1f} M".replace(".", ",")
        if abs(nilai) >= 1_000_000:
            return f"Rp{nilai / 1_000_000:.1f} jt".replace(".", ",")
        if abs(nilai) >= 1_000:
            return f"Rp{nilai / 1_000:.0f} rb"
    return "Rp{:,.0f}".format(nilai).replace(",", ".")


def simpan_grafik(nama_file: str, dpi: int = 150) -> None:
    '''Simpan grafik yang sedang aktif ke folder images/.'''
    path = FOLDER_GAMBAR / nama_file
    plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"💾 Grafik disimpan: {path}")


# Uji cepat
print(ke_rupiah(1500000))
print(ke_rupiah(45000))
print(ke_rupiah(2_350_000_000))

Rp1,5 jt
Rp45 rb
Rp2,4 M


### 1.3 Dataset yang Dipakai

Kita memakai dataset **[Tokopedia Product and Review Dataset](https://www.kaggle.com/datasets/musabiam/tokopedia-product-and-review-dataset)**:
**5.553 produk** yang diambil pada 2025, mencakup 20+ kategori, dengan lisensi
**CC BY-NC 4.0** (boleh dipakai untuk proyek non-komersial seperti portofolio, wajib
mencantumkan sumber).

Kenapa dataset ini? Karena hanya dataset ini yang punya **kolom kategori asli**.
Pertanyaan 1 dan 5 ("kategori terlaris" dan "kategori paling sensitif promo") tidak bisa
dijawab dengan jujur kalau kategorinya hasil tebakan dari nama produk.

Kolom yang akan kita pakai:

| Kolom asli | Arti | Dipakai untuk |
|---|---|---|
| `category` | Kategori & sub-kategori, dipisah tanda `|` (contoh: `dapur|peralatan_masak`) | Q1, Q2, Q5 |
| `name` | Nama produk | Konteks & contoh |
| `count_sold` | Jumlah unit terjual (proxy minat beli) | Q1–Q5 |
| `price` | Harga yang dibayar pembeli (sudah setelah diskon) | Q2, Q3 |
| `discounted_price` | Harga **sebelum** diskon (kosong kalau tidak diskon) | Q3, Q5 |
| `rating_average` | Rata-rata rating 1–5 | Q4 |
| `is_official` | Toko Official Store atau bukan | Analisis tambahan |
| `shop_location` | Lokasi toko | Analisis tambahan |
| `product_id` | ID unik produk | Cek duplikat |

**Tiga hal yang perlu kamu tahu sejak awal** (semuanya kita tangani di Tahap 2):

1. File aslinya ~345 MB karena berisi ±1 juta ulasan. Kita hanya butuh 9 kolom produk,
   jadi saat membaca nanti kita pakai `usecols` agar cepat dan hemat memori.
2. Nama kolom `discounted_price` agak menyesatkan: isinya **harga sebelum diskon**,
   bukan harga setelah diskon.
3. Data ini diambil dari halaman kategori (banyak produk "terlaris"), jadi isinya
   **cenderung produk yang sudah populer**. Ini memengaruhi cara kita membaca hasil nanti.

> Dua dataset pembanding yang tidak dipakai:
> [Tokopedia Listings — 29.519 produk](https://www.kaggle.com/datasets/pandaa12/indonesia-e-commerce-dataset-tokopedia-listings)
> (Apache 2.0, tapi tanpa kolom kategori) dan
> [Bright Data sample — 1.001 produk](https://github.com/luminati-io/Tokopedia-dataset-samples)
> (kolom lengkap, tapi lisensinya tidak dinyatakan).

### 1.4 Cara Mengunggah Dataset ke Colab

Pilih **salah satu** cara berikut. File yang kita butuhkan bernama
`tokopedia_products_with_review.csv` (~345 MB).

**Cara 1 — Upload manual (paling mudah)**

1. Unduh dataset dari Kaggle (tombol *Download* di halaman dataset).
2. Ekstrak file `.zip`-nya di komputer → akan muncul `tokopedia_products_with_review.csv`.
3. Buka Colab, klik ikon **folder** 📁 di panel kiri.
4. Klik ikon **upload** (panah ke atas), pilih file `.csv` tadi.
5. Tunggu sampai selesai (file besar, bisa 1–3 menit), lalu file muncul di
   `/content/tokopedia_products_with_review.csv`.

**Cara 2 — Lewat Google Drive (lebih stabil untuk file besar)**

```python
from google.colab import drive
drive.mount("/content/drive")
# lalu salin file ke /content supaya lebih cepat dibaca
```

**Cara 3 — Unduh otomatis dengan `kagglehub` (butuh API token)**

1. Buka https://www.kaggle.com/settings → *API* → **Create New Token**.
2. Di Colab, klik ikon 🔑 di panel kiri → tambahkan *Secret* bernama `KAGGLE_KEY`
   (isinya token tadi) dan `KAGGLE_USERNAME`.
3. Jalankan sel di bawah (hapus tanda `#`).

Sel berikut **tidak dijalankan otomatis** — hanya disiapkan kalau kamu butuh.

In [4]:
# (Opsional) Unduh dataset langsung dari Kaggle.
# Hapus tanda # di awal baris kalau ingin memakai cara ini.

# from google.colab import userdata
# import os
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
#
# import kagglehub
# path = kagglehub.dataset_download("musabiam/tokopedia-product-and-review-dataset")
# print("Dataset tersimpan di:", path)

print("ℹ️ Sel ini dilewati. Kita akan memakai file yang kamu unggah sendiri.")

ℹ️ Sel ini dilewati. Kita akan memakai file yang kamu unggah sendiri.


### 1.5 Membaca Data Pertama Kali

Sekarang kita baca file-nya. Sel di bawah akan mencari file `.csv` secara otomatis di:

1. Path yang kamu tulis manual di `FILE_DATASET`
2. Folder `/content` (lokasi default upload Colab)
3. Folder `data/raw/` (kalau kamu menjalankan notebook ini dari repo GitHub)

Kita sengaja hanya membaca **9 kolom** yang dibutuhkan (`usecols`), bukan 25 kolom.
Sisanya berisi ulasan pembeli yang tidak dipakai di analisis ini.

Kalau file belum ada, sel ini **tidak akan error** — ia hanya memberi pesan agar kamu
mengunggah file dulu.

In [5]:
# Isi path lengkap di sini kalau ingin menunjuk file tertentu.
# Contoh: FILE_DATASET = "/content/tokopedia_products_with_review.csv"
FILE_DATASET = None

# Hanya kolom ini yang dipakai sepanjang analisis
KOLOM_DIPAKAI = [
    "product_id", "category", "name", "count_sold", "discounted_price",
    "price", "rating_average", "is_official", "shop_location",
]


def cari_file_dataset():
    '''Cari file CSV dataset di beberapa lokasi yang umum.'''
    if FILE_DATASET:
        return Path(FILE_DATASET)
    for folder in (Path("/content"), Path("data/raw"), Path("../data/raw"), Path(".")):
        if folder.exists():
            for pola in ("*.csv", "*.CSV"):
                hasil = sorted(folder.glob(pola))
                if hasil:
                    return hasil[0]
    return None


path_dataset = cari_file_dataset()

if path_dataset is None:
    df = None
    print("⚠️ Dataset belum ditemukan.")
    print("   Ikuti langkah 1.4: unggah file CSV ke Colab, lalu jalankan ulang sel ini.")
else:
    print(f"⏳ Membaca {path_dataset.name} ... (file besar, butuh 30–60 detik)")
    df = pd.read_csv(path_dataset, usecols=KOLOM_DIPAKAI, low_memory=False)
    print(f"✅ Dataset terbaca: {path_dataset}")
    print(f"📦 Ukuran data : {df.shape[0]:,} baris × {df.shape[1]} kolom")

⏳ Membaca tokopedia_produk_2025.csv ... (file besar, butuh 30–60 detik)
✅ Dataset terbaca: data\raw\tokopedia_produk_2025.csv
📦 Ukuran data : 5,553 baris × 9 kolom


Sekarang kita lihat isinya: 5 baris pertama, daftar kolom, dan tipe datanya.
Ini semacam "membaca daftar isi" sebelum membedah buku — kita perlu tahu apa yang
sebenarnya kita punya sebelum mengambil kesimpulan apa pun.

In [6]:
if df is not None:
    print("=== 5 BARIS PERTAMA ===")
    display(df.head())

    print("\n=== DAFTAR KOLOM & TIPE DATA ===")
    ringkasan = pd.DataFrame({
        "kolom"        : df.columns,
        "tipe_data"    : [str(t) for t in df.dtypes],
        "jumlah_kosong": df.isna().sum().values,
        "contoh_nilai" : [df[c].dropna().iloc[0] if df[c].notna().any() else "-" for c in df.columns],
    })
    ringkasan["contoh_nilai"] = ringkasan["contoh_nilai"].astype(str).str.slice(0, 45)
    display(ringkasan)
else:
    print("(Belum ada data untuk ditampilkan.)")

=== 5 BARIS PERTAMA ===


,product_id,category,name,count_sold,discounted_price,price,is_official,rating_average,shop_location
0,268823794,rumah_tangga|storage_fair,Ambalan Laci Melayang 60x20x5.5 MERK KINBAR A543,"2,112.00",NaN,108500,False,4.80,Kota Administrasi Jakarta Barat
1,2145556006,rumah_tangga|storage_fair,Daiso Storage Box with Lid -,285.00,NaN,33000,True,5.00,Kota Administrasi Jakarta Selatan
2,1276550898,rumah_tangga|storage_fair,Shinpo 204 Kotak Box Tempat Masker Pensil Peny...,"3,686.00",NaN,9300,False,4.90,Kota Administrasi Jakarta Barat
3,781949458,rumah_tangga|storage_fair,Informa 42X20X79 cm Rak Sepatu 5 Tingkat - Hit...,"5,354.00","128,600.00",59400,True,4.70,Kota Administrasi Jakarta Utara
4,9665251383,rumah_tangga|storage_fair,STORA LACI PENYIMPANAN STACKABLE - BIRU TUA,97.00,"32,300.00",21700,True,5.00,Kota Administrasi Jakarta Selatan



=== DAFTAR KOLOM & TIPE DATA ===


,kolom,tipe_data,jumlah_kosong,contoh_nilai
0,product_id,int64,0,268823794
1,category,str,0,rumah_tangga|storage_fair
2,name,str,0,Ambalan Laci Melayang 60x20x5.5 MERK KINBAR A
3,count_sold,float64,810,2112.0
4,discounted_price,float64,4213,128600.0
5,price,int64,0,108500
6,is_official,bool,0,False
7,rating_average,float64,66,4.8
8,shop_location,str,27,Kota Administrasi Jakarta Barat


✅ **Tahap 1 selesai.**

Yang sudah kita punya:
- Library terpasang & tampilan grafik sudah seragam
- Dataset terbaca, dan kita tahu bentuk serta tipe kolomnya

**Yang kita temukan sekilas dari data ini** (dibahas tuntas di Tahap 2):
- `count_sold` sudah berupa angka, tapi **±15% kosong** — perlu diputuskan mau diapakan.
- `category` berbentuk `kategori|subkategori`, jadi jumlah "kategori" mentahnya terlihat
  lebih banyak daripada kategori sebenarnya.
- Ada produk yang **muncul lebih dari sekali** (duplikat).
- `rating_average` hampir semuanya 5.0 — ini akan berpengaruh besar ke Pertanyaan 4.

👉 **Lanjut ke Tahap 2: Memahami & Membersihkan Data** — di sana kita rapikan semua
ketidakberesan di atas, satu per satu, dengan alasan yang jelas untuk setiap keputusan.